Objectif du notebook n°4 : 

L'objectif de ce notebook est d'enrichir notre base de données de films, initialement limitée aux genres (MovieLens), avec des informations complémentaires provenant de l'API TMDb (The Movie Database) : synopsis, réalisateur, acteurs principaux et mots-clés thématiques. Cet enrichissement permettra, dans la suite du projet, de construire un profil utilisateur plus riche et plus précis à partir des films qu'il sélectionne, en dépassant les limites du modèle content-based basé uniquement sur les genres, mises en évidence lors de notre phase d'évaluation (notebook n°3). Pour effectuer la correspondance entre nos films MovieLens et leur identifiant TMDb, nous utilisons le fichier links_clean.csv, qui contient un identifiant tmdbId pour chaque film de notre catalogue.

In [22]:
import requests

cle_api = "656d87c4d7ec10f2e6a6b079d581bf2c"


Avant d'automatiser la récupération des données pour tous nos films, nous testons l'API sur un seul film connu (Toy Story) afin de vérifier que notre clé API fonctionne correctement et de comprendre la structure des données renvoyées. Cet appel simple retourne les informations générales du film : synopsis, genres selon TMDb, date de sortie, société de production, etc.

In [23]:
# On charge les fichiers nettoyés dont on a besoin pour l'analyse

import pandas as pd

# On charge les fichiers nettoyés dont on a besoin pour ce notebook
films = pd.read_csv('/Users/nazmanazirhussain/Desktop/RecommandationFilm/data/nettoye/movies_clean.csv')
links = pd.read_csv('/Users/nazmanazirhussain/Desktop/RecommandationFilm/data/nettoye/links_clean.csv')

print("films :", films.shape)
print("links :", links.shape)

films : (9742, 6)
links : (9742, 3)


In [24]:
# On récupère l'identifiant TMDb de Toy Story (movieId = 1 dans MovieLens)
# en utilisant le fichier links_clean.csv, qui fait la correspondance entre les deux bases
ligne_toy_story = links[links['movieId'] == 1]
identifiant_tmdb = int(ligne_toy_story['tmdbId'].values[0])

print("Identifiant TMDb de Toy Story :", identifiant_tmdb)

Identifiant TMDb de Toy Story : 862


In [25]:
adresse_credits = f"https://api.themoviedb.org/3/movie/{identifiant_tmdb}/credits?api_key={cle_api}&language=fr-FR"
reponse_credits = requests.get(adresse_credits)

print("Code de statut :", reponse_credits.status_code)

Code de statut : 200


Étape 1 : Extraire le réalisateur et les acteurs (fonction)

In [26]:
def extraire_realisateur_et_acteurs(donnees_credits, nombre_acteurs=3):
    # On cherche dans l'équipe (crew) la personne dont le poste est "Director"
    nom_realisateur = None
    
    for personne in donnees_credits['crew']:
        if personne['job'] == 'Director' and nom_realisateur is None:
            nom_realisateur = personne['name']
    
    # On récupère les premiers acteurs de la liste (déjà triée par ordre d'importance)
    liste_acteurs = [acteur['name'] for acteur in donnees_credits['cast'][:nombre_acteurs]]
    
    return nom_realisateur, liste_acteurs

In [27]:
donnees_credits = reponse_credits.json()
nom_realisateur, liste_acteurs = extraire_realisateur_et_acteurs(donnees_credits)

print("Réalisateur :", nom_realisateur)
print("Acteurs principaux :", liste_acteurs)

Réalisateur : John Lasseter
Acteurs principaux : ['Tom Hanks', 'Tim Allen', 'Don Rickles']


Étape 2 : Construire une seule fonction qui récupère tout pour un film donné 

In [28]:
def recuperer_infos_film(id_tmdb):
    # Appel 1 : informations générales (synopsis, genres, date...)
    adresse_infos = f"https://api.themoviedb.org/3/movie/{id_tmdb}?api_key={cle_api}&language=fr-FR"
    reponse_infos = requests.get(adresse_infos)
    
    # Appel 2 : acteurs et réalisateur
    adresse_credits = f"https://api.themoviedb.org/3/movie/{id_tmdb}/credits?api_key={cle_api}&language=fr-FR"
    reponse_credits = requests.get(adresse_credits)
    
    if reponse_infos.status_code != 200 or reponse_credits.status_code != 200:
        return None
    
    donnees_infos = reponse_infos.json()
    donnees_credits = reponse_credits.json()
    
    synopsis = donnees_infos.get('overview', '')
    
    nom_realisateur, liste_acteurs = extraire_realisateur_et_acteurs(donnees_credits)
    
    return {
        'synopsis': synopsis,
        'realisateur': nom_realisateur,
        'acteurs': liste_acteurs
    }

Étape 3 : Tester cette fonction complète sur un seul film 

In [29]:
infos_toy_story = recuperer_infos_film(identifiant_tmdb)
print(infos_toy_story)

{'synopsis': "Dans un monde où les jouets vivent leur vie quand les humains ne sont pas présents, Toy Story emmène les spectateurs dans un voyage fantastique vu principalement par les yeux de deux rivaux : Woody, un cow-boy, et Buzz l'Éclair, un ranger de l’espace. Ce duo va devoir apprendre à mettre ses différences de côté et s'allier, lorsqu’il sera séparé de son propriétaire Andy.", 'realisateur': 'John Lasseter', 'acteurs': ['Tom Hanks', 'Tim Allen', 'Don Rickles']}


Étape 4 : Tester d'abord sur un petit échantillon 

In [30]:
import time

# On prend les 20 premiers films pour tester
echantillon_films = films.head(20).copy()

resultats = []

for index, ligne in echantillon_films.iterrows():
    # On récupère l'identifiant TMDb correspondant à ce film via links
    ligne_correspondance = links[links['movieId'] == ligne['movieId']]
    
    if len(ligne_correspondance) == 0 or pd.isnull(ligne_correspondance['tmdbId'].values[0]):
        continue  # pas d'identifiant TMDb trouvé pour ce film, on passe au suivant
    
    id_tmdb = int(ligne_correspondance['tmdbId'].values[0])
    
    infos = recuperer_infos_film(id_tmdb)
    
    if infos is not None:
        infos['movieId'] = ligne['movieId']
        resultats.append(infos)
    
    # Petite pause pour ne pas surcharger l'API
    time.sleep(0.1)

print("Nombre de films récupérés :", len(resultats))
resultats[:3]

Nombre de films récupérés : 20


[{'synopsis': "Dans un monde où les jouets vivent leur vie quand les humains ne sont pas présents, Toy Story emmène les spectateurs dans un voyage fantastique vu principalement par les yeux de deux rivaux : Woody, un cow-boy, et Buzz l'Éclair, un ranger de l’espace. Ce duo va devoir apprendre à mettre ses différences de côté et s'allier, lorsqu’il sera séparé de son propriétaire Andy.",
  'realisateur': 'John Lasseter',
  'acteurs': ['Tom Hanks', 'Tim Allen', 'Don Rickles'],
  'movieId': 1},
 {'synopsis': "Lors d'une partie de Jumanji, un jeu très ancien, le jeune Alan est propulsé sous les yeux de son amie d'enfance, Sarah, dans un étrange pays. Il ne pourra s'en échapper que lorsqu'un autre joueur reprendra la partie et le libèrera sur un coup de dés. Vingt-six ans plus tard, il retrouve le monde réel par le coup de dés de deux autres jeunes joueurs.",
  'realisateur': 'Joe Johnston',
  'acteurs': ['Robin Williams', 'Kirsten Dunst', 'Bradley Pierce'],
  'movieId': 2},
 {'synopsis': "

Étape 5 : Préparer un fichier de sauvegarde progressive

In [31]:
import os

# Chemin du fichier où on va sauvegarder au fur et à mesure
chemin_sauvegarde = '/Users/nazmanazirhussain/Desktop/RecommandationFilm/data/enrichi/films_tmdb.csv'

# On crée le dossier s'il n'existe pas déjà
os.makedirs(os.path.dirname(chemin_sauvegarde), exist_ok=True)

Étape 6 : Vérifier si un fichier de sauvegarde existe déjà 

In [32]:
if os.path.exists(chemin_sauvegarde):
    resultats_deja_faits = pd.read_csv(chemin_sauvegarde)
    films_deja_traites = set(resultats_deja_faits['movieId'].tolist())
    print("Reprise : ", len(films_deja_traites), "films déjà enrichis précédemment")
else:
    resultats_deja_faits = pd.DataFrame(columns=['movieId', 'synopsis', 'realisateur', 'acteurs'])
    films_deja_traites = set()
    print("Aucune sauvegarde existante, on démarre depuis le début")

Reprise :  9621 films déjà enrichis précédemment


Étape 7 : Boucle principale avec sauvegarde tous les 200 films

In [33]:
import time

resultats_temporaires = []
compteur = 0

for index, ligne in films.iterrows():
    # Si ce film a déjà été traité lors d'une exécution précédente, on le saute
    if ligne['movieId'] in films_deja_traites:
        continue
    
    # On récupère l'identifiant TMDb correspondant
    ligne_correspondance = links[links['movieId'] == ligne['movieId']]
    
    if len(ligne_correspondance) == 0 or pd.isnull(ligne_correspondance['tmdbId'].values[0]):
        continue
    
    id_tmdb = int(ligne_correspondance['tmdbId'].values[0])
    
    infos = recuperer_infos_film(id_tmdb)
    
    if infos is not None:
        infos['movieId'] = ligne['movieId']
        resultats_temporaires.append(infos)
    
    compteur = compteur + 1
    
    # Toutes les 200 films, on sauvegarde ce qu'on a accumulé jusque-là
    if compteur % 200 == 0:
        nouveau_lot = pd.DataFrame(resultats_temporaires)
        resultats_deja_faits = pd.concat([resultats_deja_faits, nouveau_lot], ignore_index=True)
        resultats_deja_faits.to_csv(chemin_sauvegarde, index=False)
        resultats_temporaires = []
        print(f"{compteur} films traités dans cette session, sauvegarde effectuée")
    
    time.sleep(0.1)

# Sauvegarde finale pour les derniers films (moins de 200 depuis la dernière sauvegarde)
if len(resultats_temporaires) > 0:
    nouveau_lot = pd.DataFrame(resultats_temporaires)
    resultats_deja_faits = pd.concat([resultats_deja_faits, nouveau_lot], ignore_index=True)
    resultats_deja_faits.to_csv(chemin_sauvegarde, index=False)

print("Terminé ! Nombre total de films enrichis :", len(resultats_deja_faits))

Terminé ! Nombre total de films enrichis : 9621


On a réussi à obtenir les informations concernant 9621 films. Mais il nous manque encore 121 films. On va donc voir ce qu'il s'est passé avec ces films. 

In [34]:
# Films MovieLens qui n'ont pas d'identifiant TMDb dans links.csv
films_sans_tmdb_id = links[links['tmdbId'].isnull()]
print("Films sans identifiant TMDb dans links.csv :", len(films_sans_tmdb_id))

Films sans identifiant TMDb dans links.csv : 8


Sur les 121 films manquants, seulement 8 s'expliquent par l'absence d'identifiant TMDb. Il reste 113 films à comprendre. 

In [35]:
# Films dans movies_clean.csv qui ne sont PAS dans notre fichier enrichi final
resultats_finaux = pd.read_csv('/Users/nazmanazirhussain/Desktop/RecommandationFilm/data/enrichi/films_tmdb.csv')
films_manquants = films[~films['movieId'].isin(resultats_finaux['movieId'])]

print("Nombre de films manquants :", len(films_manquants))
films_manquants[['titre', 'genres', 'annee']].head(20)

Nombre de films manquants : 121


,titre,genres,annee
624,"Last Klezmer: Leopold Kozlowski, His Life and ...",Documentary,1994
843,Loser,Comedy,1991
2141,Saturn 3,Adventure|Sci-Fi|Thriller,1980
3027,Horrors of Spider Island (Ein Toter Hing im Netz),Horror|Sci-Fi,1960
3127,Navy Seals,Action|Adventure|War,1990
3362,Best of the Best,Action,1989
3680,Escaflowne: The Movie (Escaflowne),Action|Adventure|Animation|Drama|Fantasy,2000
3741,Ffolkes,Action|Adventure|Thriller,1979
4981,Rose Red,Horror|Mystery|Thriller,2002
4986,Pride and Prejudice,Drama|Romance,1995


In [36]:
# On regarde les films manquants qui AVAIENT pourtant un identifiant TMDb valide
films_manquants_avec_id = films_manquants[~films_manquants['movieId'].isin(films_sans_tmdb_id['movieId'])]

print("Films avec un identifiant TMDb valide mais absents du résultat :", len(films_manquants_avec_id))
films_manquants_avec_id[['movieId', 'titre', 'annee']].head(20)

Films avec un identifiant TMDb valide mais absents du résultat : 113


,movieId,titre,annee
3127,4207,Navy Seals,1990
3362,4568,Best of the Best,1989
3680,5069,Escaflowne: The Movie (Escaflowne),2000
3741,5209,Ffolkes,1979
4981,7646,Rose Red,2002
4986,7669,Pride and Prejudice,1995
5011,7762,"Tinker, Tailor, Soldier, Spy",1979
5036,7841,Children of Dune,2003
5037,7842,Dune,2000
5509,26453,Smiley's People,1982


In [37]:
# Remplacez 99999 par un movieId réel trouvé dans films_manquants_avec_id
movie_id_test = films_manquants_avec_id['movieId'].iloc[0]

ligne_correspondance = links[links['movieId'] == movie_id_test]
id_tmdb_test = int(ligne_correspondance['tmdbId'].values[0])

adresse_test = f"https://api.themoviedb.org/3/movie/{id_tmdb_test}?api_key={cle_api}&language=fr-FR"
reponse_test = requests.get(adresse_test)

print("Titre MovieLens :", films_manquants_avec_id['titre'].iloc[0])
print("Identifiant TMDb testé :", id_tmdb_test)
print("Code de statut :", reponse_test.status_code)
print(reponse_test.json())

Titre MovieLens : Navy Seals
Identifiant TMDb testé : 12773
Code de statut : 404
{'success': False, 'status_code': 34, 'status_message': 'The resource you requested could not be found.'}


Commentaire : Sur les 9742 films de notre catalogue, 9621 ont été enrichis avec succès (98,8%). Les 121 films manquants se répartissent en deux catégories : 8 films ne possédaient aucun identifiant TMDb dans links_clean.csv, et 113 films possédaient un identifiant TMDb qui s'est révélé invalide (erreur 404 lors de l'appel API), probablement en raison d'un désynchronisation entre la version de links.csv fournie par MovieLens et l'état actuel de la base TMDb. Ces 121 films resteront exploitables pour les modèles content-based (genres) et collaboratif, qui ne dépendent pas des données TMDb, mais ne bénéficieront pas de l'enrichissement synopsis/acteurs/réalisateur dans la suite du projet.

Étape 8 : Fusion du fichier "films_tmdb.csv" avec le fichier "movie_clean.csv"

In [38]:
# On fusionne les données enrichies avec le tableau principal des films
films_complet = films.merge(resultats_finaux, on='movieId', how='left')

print("Dimensions du tableau final :", films_complet.shape)
films_complet.head()

Dimensions du tableau final : (9742, 9)


,movieId,title,genres,annee,titre,genres_list,synopsis,realisateur,acteurs
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995,Toy Story,"['Adventure', 'Animation', 'Children', 'Comedy...",Dans un monde où les jouets vivent leur vie qu...,John Lasseter,"['Tom Hanks', 'Tim Allen', 'Don Rickles']"
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995,Jumanji,"['Adventure', 'Children', 'Fantasy']","Lors d'une partie de Jumanji, un jeu très anci...",Joe Johnston,"['Robin Williams', 'Kirsten Dunst', 'Bradley P..."
2,3,Grumpier Old Men (1995),Comedy|Romance,1995,Grumpier Old Men,"['Comedy', 'Romance']","Après le mariage de John et d'Ariel, Max se re...",Howard Deutch,"['Walter Matthau', 'Jack Lemmon', 'Ann-Margret']"
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995,Waiting to Exhale,"['Comedy', 'Drama', 'Romance']",L'amitié de quatre femmes qui tentent de surmo...,Forest Whitaker,"['Whitney Houston', 'Angela Bassett', 'Loretta..."
4,5,Father of the Bride Part II (1995),Comedy,1995,Father of the Bride Part II,['Comedy'],George Banks et sa petite famille sont de reto...,Charles Shyer,"['Steve Martin', 'Diane Keaton', 'Martin Short']"


In [39]:
films_complet.to_csv('/Users/nazmanazirhussain/Desktop/RecommandationFilm/data/enrichi/films_complet.csv', index=False)

In [40]:
import os
print(os.path.exists('/Users/nazmanazirhussain/Desktop/RecommandationFilm/data/enrichi/films_complet.csv'))

True


Le fichier "films_complet.csv" existe bien dans le dossier data/enrichi. 

Étape 8 : Vérification approfondie de la qualité des données enrichies

In [41]:
resultats_bruts = pd.read_csv('/Users/nazmanazirhussain/Desktop/RecommandationFilm/data/enrichi/films_tmdb.csv')

print("Nombre total de films dans ce fichier :", len(resultats_bruts))
print("Synopsis NaN :", resultats_bruts['synopsis'].isnull().sum())
print("Synopsis vides ('') :", (resultats_bruts['synopsis'] == '').sum())

Nombre total de films dans ce fichier : 9621
Synopsis NaN : 609
Synopsis vides ('') : 0


Comprendre le nombre de synopsis manquants (730)

Le nombre de synopsis manquants dans le fichier final (films_complet.csv) s'élève à 730, ce qui s'explique par l'addition de deux causes distinctes : les 121 films sans correspondance TMDb (détaillés précédemment), et 609 films supplémentaires bien référencés sur TMDb mais pour lesquels le champ "overview" était vide (null) côté API. Ce cas se produit lorsque TMDb possède une fiche pour le film sans description textuelle associée, généralement pour des films peu connus. Ces 730 films au total seront traités comme des synopsis vides lors de la construction du modèle content-based enrichi (notebook n°5), sans impact sur les autres informations disponibles (genres, réalisateur, acteurs).

In [42]:
print("Réalisateur manquant (NaN) :", resultats_bruts['realisateur'].isnull().sum())
print("Acteurs manquant (NaN) :", resultats_bruts['acteurs'].isnull().sum())
print()

# Cas 1 : synopsis présent, mais réalisateur manquant
cas_1 = resultats_bruts[(resultats_bruts['synopsis'] != '') & (resultats_bruts['realisateur'].isnull())]
print("Synopsis présent mais réalisateur manquant :", len(cas_1))

# Cas 2 : réalisateur présent, mais synopsis vide
cas_2 = resultats_bruts[(resultats_bruts['synopsis'] == '') & (resultats_bruts['realisateur'].notnull())]
print("Réalisateur présent mais synopsis vide :", len(cas_2))

# Cas 3 : les deux manquent en même temps
cas_3 = resultats_bruts[(resultats_bruts['synopsis'] == '') & (resultats_bruts['realisateur'].isnull())]
print("Synopsis ET réalisateur manquants ensemble :", len(cas_3))

Réalisateur manquant (NaN) : 4
Acteurs manquant (NaN) : 0

Synopsis présent mais réalisateur manquant : 4
Réalisateur présent mais synopsis vide : 0
Synopsis ET réalisateur manquants ensemble : 0


Conclusion : 

Ce notebook a permis d'enrichir notre catalogue de films avec des données complémentaires issues de l'API TMDb : synopsis, réalisateur et acteurs principaux. Sur les 9742 films du catalogue MovieLens, 9621 ont pu être associés avec succès à l'API TMDb (98,8%), les 121 films restants étant soit dépourvus d'identifiant TMDb valide, soit non retrouvés dans la base actuelle de TMDb. Parmi les films enrichis, la quasi-totalité des données récupérées sont complètes : les acteurs sont disponibles pour 100% des films, le réalisateur pour 99,96% d'entre eux, et le synopsis pour 93,7% (609 films disposant d'une fiche TMDb sans description associée). Le fichier final, films_complet.csv, combine ces nouvelles informations avec les données nettoyées du notebook n°1 (genres, année) et servira de base, dans le notebook suivant, à la construction d'un modèle de recommandation content-based enrichi.
